# Core reproduction

From the compact 19-column object, this freshly learns development-only preprocessing, fits exactly M0, M_any, M_count and M1 with R/Efron, recomputes point results, and generates 500 fresh paired BIN loss resamples. Non-loss primary intervals are explicitly reconstructed from authentic retained arrays. No raw source or fitted model is included.


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, time
from google.colab import files
START = time.perf_counter()
ROOT = Path("/content/amendment-history-monitoring")
if not ROOT.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/evidenceworks/amendment-history-monitoring.git", str(ROOT)], check=True)
print({"repository": str(ROOT)})


In [ ]:
subprocess.run(["bash", "environment/setup_python.sh"], cwd=ROOT, check=True)
CONTROLLED_PYTHON = Path("/content/reproduction-python-3.12.10/bin/python")
if not CONTROLLED_PYTHON.is_file(): raise RuntimeError("Controlled Python was not provisioned")
env = dict(os.environ, REPRO_PYTHON=str(CONTROLLED_PYTHON), OPENBLAS_NUM_THREADS="1", OMP_NUM_THREADS="1")
subprocess.run(["bash", "environment/setup_r.sh"], cwd=ROOT, check=True, env=env)
subprocess.run([str(CONTROLLED_PYTHON), "verification/verify_science.py"], cwd=ROOT, check=True, env=env)
subprocess.run([str(CONTROLLED_PYTHON), "scripts/check_runtime.py", "--rscript", "Rscript"], cwd=ROOT, check=True, env=env)


In [ ]:
OUTPUT = Path("/content/core_reproduction_output")
if OUTPUT.exists(): shutil.rmtree(OUTPUT)
subprocess.run([str(CONTROLLED_PYTHON), "scripts/run_core.py", "--output", str(OUTPUT), "--rscript", "Rscript"], cwd=ROOT, check=True, env=env)
subprocess.run([str(CONTROLLED_PYTHON), "verification/verify_core.py", "--output", str(OUTPUT), "--json-out", str(OUTPUT/"final_core_verification.json")], cwd=ROOT, check=True, env=env)
elapsed = time.perf_counter()-START
if elapsed > 900: raise RuntimeError(f"Core runtime ceiling exceeded: {elapsed:.3f}s")
receipt={"status":"CORE_REPRODUCTION_PASS","transport":"repository clone or existing checkout","total_seconds":elapsed,"platform":platform.platform(),"python":subprocess.check_output([str(CONTROLLED_PYTHON),"--version"],text=True).strip(),"r":subprocess.check_output(["Rscript","-e","cat(as.character(getRversion()),' ',as.character(packageVersion('survival')))"] ,text=True).strip(),"compact_sha256":json.loads((ROOT/"data/provenance.json").read_text())["compact_sha256"],"scientific_comparison":json.loads((OUTPUT/"results/scientific_comparison.json").read_text())}
receipt_path=OUTPUT/"core_reproduction_receipt.json";receipt_path.write_text(json.dumps(receipt,indent=2)+"\n")
print(json.dumps(receipt,indent=2))


In [ ]:
archive=shutil.make_archive("/content/core_reproduction_results","zip",OUTPUT)
files.download(archive)
